# risk-management_002

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (123).ipynb`

| Field | Value |
|---|---|
| Section | `risk_management` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 10 |
| Detected functions | calculate_strategy, process_tickers, calculate_supertrend, strategy_4A, test_combination, robustness_testing |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import pandas as pd

# Use a raw string or replace backslashes with forward slashes
data = pd.read_parquet('AUG/AUG_2024.parquet')  # Raw string

# Display the DataFrame
data


In [ ]:
import pandas as pd

# Assuming your DataFrame is initially named df
data = data.dropna(subset=['LTP', 'BuyPrice', 'SellPrice'])  # Drop rows with NaN values
data = data[(data['LTP'] != 0) & (data['BuyPrice'] != 0) & (data['SellPrice'] != 0)]  # Drop rows with 0 values

# Display the cleaned DataFrame
print(data.head())


In [ ]:
data.Ticker.unique()

In [ ]:
import pandas as pd
import time
from tqdm import tqdm  # Import tqdm for progress bar

def calculate_strategy(group, threshold, trailing_stop_n, trailing_stop_m, capital_per_trade, brokerage_percent):
    """
    Implements the strategy for a single group (ticker) with brokerage and optimized calculations.
    """
    trades = []
    position_status = None  # Tracks current position ('long', 'short', None)
    entry_price, shares, trailing_stop = None, None, None

    group = group.sort_values('Date_Time')  # Ensure chronological order

    for i in range(1, len(group)):
        # Fetch current and previous prices
        current_row = group.iloc[i]
        previous_row = group.iloc[i - 1]
        current_time = current_row['Date_Time']
        current_buy_price = current_row['BuyPrice']
        current_sell_price = current_row['SellPrice']

        # Open a long position
        if position_status is None and current_buy_price >= previous_row['BuyPrice'] * (1 + threshold):
            position_status = 'long'
            entry_price = current_buy_price
            shares = capital_per_trade / entry_price
            trailing_stop = current_buy_price * (1 - threshold / trailing_stop_n)
            trades.append({'entry_time': current_time, 'entry_price': entry_price, 'type': 'long', 'shares': shares})

        # Open a short position
        elif position_status is None and current_sell_price <= previous_row['SellPrice'] * (1 - threshold):
            position_status = 'short'
            entry_price = current_sell_price
            shares = capital_per_trade / entry_price
            trailing_stop = current_sell_price * (1 + threshold / trailing_stop_n)
            trades.append({'entry_time': current_time, 'entry_price': entry_price, 'type': 'short', 'shares': shares})

        # Update trailing stop based on the current position
        if position_status == 'long':
            trailing_stop = max(trailing_stop, current_buy_price * (1 - threshold / trailing_stop_n))
        elif position_status == 'short':
            trailing_stop = min(trailing_stop, current_sell_price * (1 + threshold / trailing_stop_n))

        # Close a long position
        if position_status == 'long' and (current_sell_price <= trailing_stop or current_time.time() >= pd.Timestamp("15:28").time()):
            raw_pnl = (current_sell_price - entry_price) * shares
            trade_value = (entry_price * shares) + (current_sell_price * shares)
            brokerage = trade_value * (brokerage_percent / 100)
            pnl = raw_pnl - brokerage
            trades[-1].update({'exit_time': current_time, 'exit_price': current_sell_price, 'PnL': pnl})

            # Reset position
            position_status, entry_price, shares, trailing_stop = None, None, None, None

        # Close a short position
        elif position_status == 'short' and (current_buy_price >= trailing_stop or current_time.time() >= pd.Timestamp("15:28").time()):
            raw_pnl = (entry_price - current_buy_price) * shares
            trade_value = (entry_price * shares) + (current_buy_price * shares)
            brokerage = trade_value * (brokerage_percent / 100)
            pnl = raw_pnl - brokerage
            trades[-1].update({'exit_time': current_time, 'exit_price': current_buy_price, 'PnL': pnl})

            # Reset position
            position_status, entry_price, shares, trailing_stop = None, None, None, None

    return pd.DataFrame(trades)

def process_tickers(data, threshold, trailing_stop_n, trailing_stop_m, capital_per_trade, brokerage_percent):
    """
    Process all tickers using the strategy and compile results into a single DataFrame.
    """
    results = []
    grouped = data.groupby('Ticker')

    # Add progress bar for large datasets
    for ticker, group in tqdm(grouped, desc="Processing Tickers", total=len(grouped)):
        ticker_trades = calculate_strategy(
            group.reset_index(drop=True),
            threshold,
            trailing_stop_n,
            trailing_stop_m,
            capital_per_trade,
            brokerage_percent
        )
        if not ticker_trades.empty:
            ticker_trades['Ticker'] = ticker
            ticker_trades['PnL_Percent'] = (ticker_trades['PnL'] / capital_per_trade) * 100
            results.append(ticker_trades)

    combined_trades = pd.concat(results, ignore_index=True)

    # Compute cumulative PnL
    combined_trades['Cumulative_PnL'] = combined_trades['PnL'].cumsum()
    return combined_trades

# Configuration Parameters
threshold = 0.009
trailing_stop_n = 1
trailing_stop_m = 0.25
capital_per_trade = 100000  # ₹1 lakh per trade
brokerage_percent = 0.02  # 0.02% brokerage fee

# Example: Assuming data is a preloaded DataFrame with the necessary columns
# data = pd.read_csv("your_data.csv", parse_dates=['Date_Time'])

# Start measuring execution time
start_time = time.time()

# Process all tickers and compute trades
optimized_trades = process_tickers(
    data, 
    threshold, 
    trailing_stop_n, 
    trailing_stop_m, 
    capital_per_trade, 
    brokerage_percent
)

# End measuring execution time
end_time = time.time()

# Print execution time
execution_time = end_time - start_time
print(f"Execution Time: {execution_time:.2f} seconds")

# Save results
optimized_trades.to_csv("optimized_trades_with_brokerage.csv", index=False)

# Summary Analysis
print(f"Total PnL: ₹{optimized_trades['PnL'].sum():,.2f}")
print(f"Total Cumulative PnL: ₹{optimized_trades['Cumulative_PnL'].iloc[-1]:,.2f}")
print(f"Total PnL as Percentage: {optimized_trades['PnL_Percent'].sum():,.2f}%")
print(optimized_trades.head())


In [ ]:
optimized_trades.columns

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

# Assuming the data is in a pandas DataFrame called df
df = pd.read_csv('optimized_trades_with_brokerage.csv')

# Convert 'entry_time' and 'exit_time' to datetime, auto-detect the format
df['entry_time'] = pd.to_datetime(df['entry_time'], errors='coerce').dt.time
df['exit_time'] = pd.to_datetime(df['exit_time'], errors='coerce').dt.time

# Plot 1: Cumulative profit over time for each ticker
fig1 = go.Figure()
for ticker in df['Ticker'].unique():
    ticker_data = df[df['Ticker'] == ticker]
    fig1.add_trace(go.Scatter(x=ticker_data['exit_time'], 
                             y=ticker_data['Cumulative_PnL'], 
                             mode='lines', 
                             name=ticker))
fig1.update_layout(title='Cumulative Profit Over Time by Ticker',
                  xaxis_title='Time', 
                  yaxis_title='Cumulative PnL ($)', 
                  template='plotly_dark',
                  width=1200, height=600)  # Increased width
fig1.show()

# Plot 2: Drawdown over time for each ticker
fig2 = go.Figure()
for ticker in df['Ticker'].unique():
    ticker_data = df[df['Ticker'] == ticker]
    ticker_data['Portfolio_Value'] = ticker_data['Cumulative_PnL'] + 100000  # Assuming initial capital of $100,000
    ticker_data['drawdown'] = ticker_data['Portfolio_Value'] - ticker_data['Portfolio_Value'].cummax()
    fig2.add_trace(go.Scatter(x=ticker_data['exit_time'], 
                             y=ticker_data['drawdown'], 
                             mode='lines', 
                             name=ticker))
fig2.update_layout(title='Drawdown Over Time by Ticker',
                  xaxis_title='Time', 
                  yaxis_title='Drawdown ($)', 
                  template='plotly_dark',
                  width=1200, height=600)  # Increased width
fig2.show()

# Plot 3: Total Profit Percentage by Ticker (Bar chart)
total_profit_percent_by_ticker = []

# Calculate total profit percentage for each ticker
for ticker in df['Ticker'].unique():
    ticker_data = df[df['Ticker'] == ticker]
    total_profit_percent = (ticker_data['PnL_Percent'].sum())  # Sum of profit percentage
    total_profit_percent_by_ticker.append(total_profit_percent)

fig3 = px.bar(x=df['Ticker'].unique(), y=total_profit_percent_by_ticker, 
              labels={'x': 'Ticker', 'y': 'Total Profit Percentage (%)'}, 
              title='Total Profit Percentage by Ticker', 
              template='plotly_dark')
fig3.update_traces(marker=dict(color='skyblue', opacity=0.7))
fig3.update_layout(width=1200, height=600)  # Increased width
fig3.show()

# Plot 4: PnL distribution (Histogram)
fig4 = px.histogram(df, x='PnL', nbins=20, title='PnL Distribution', 
                    labels={'PnL': 'PnL ($)'}, 
                    template='plotly_dark')
fig4.update_traces(marker=dict(color='purple', opacity=0.75))
fig4.update_layout(width=1200, height=600)  # Increased width
fig4.show()

# Plot 5: Scatter plot of entry and exit times (with colors for entry and exit)
fig5 = go.Figure()

# Loop through each ticker
for ticker in df['Ticker'].unique():
    ticker_data = df[df['Ticker'] == ticker]
    
    # Format the entry and exit times to HH:MM:SS:MS
    ticker_data['entry_time_str'] = ticker_data['entry_time'].astype(str)
    ticker_data['exit_time_str'] = ticker_data['exit_time'].astype(str)
    
    # Entry time (x-axis, blue color)
    fig5.add_trace(go.Scatter(x=ticker_data['entry_time_str'], 
                             y=ticker_data['exit_time_str'], 
                             mode='markers', 
                             name=f'{ticker} Entry Time', 
                             marker=dict(color='blue', size=8, opacity=0.6), 
                             text=ticker_data['entry_time_str']))
    
    # Exit time (x-axis, red color)
    fig5.add_trace(go.Scatter(x=ticker_data['exit_time_str'], 
                             y=ticker_data['exit_time_str'], 
                             mode='markers', 
                             name=f'{ticker} Exit Time', 
                             marker=dict(color='red', size=8, opacity=0.6), 
                             text=ticker_data['exit_time_str']))
    
fig5.update_layout(
    title='Entry Time vs Exit Time', 
    xaxis_title='Entry Time (HH:MM:SS)', 
    yaxis_title='Exit Time (HH:MM:SS)', 
    template='plotly_dark', 
    showlegend=True,
    width=1200, 
    height=600  # Increased width
)

fig5.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# Assuming the data is in a pandas DataFrame called df
df = pd.read_csv('optimized_trades_with_brokerage.csv')

# Create a PDF file to save the plots
with PdfPages('trading_plots.pdf') as pdf:

    # Plotting 'Cumulative_PnL' column
    plt.figure(figsize=(12, 6))
    plt.plot(df['Cumulative_PnL'])
    plt.title('Cumulative Profit and Loss Over Time')
    plt.xlabel('Time')
    plt.ylabel('Cumulative PnL ($)')
    pdf.savefig()  # Save the current figure to the PDF
    plt.close()

    # Calculate total profit percentage for each ticker
    total_profit_percent_by_ticker = []
    for ticker in df['Ticker'].unique():
        ticker_data = df[df['Ticker'] == ticker]
        total_profit_percent = ticker_data['PnL_Percent'].sum()  # Sum of profit percentage
        total_profit_percent_by_ticker.append(total_profit_percent)

    # Convert to a DataFrame for sorting and better control
    ticker_summary = pd.DataFrame({
        'Ticker': df['Ticker'].unique(),
        'Total_Profit_Percent': total_profit_percent_by_ticker
    }).sort_values(by='Total_Profit_Percent', ascending=False)

    # Create horizontal bar chart with thinner bars
    plt.figure(figsize=(14, 24))  # Adjusted height for 50 tickers
    plt.barh(ticker_summary['Ticker'], ticker_summary['Total_Profit_Percent'], color='skyblue', alpha=0.7, height=0.6)
    plt.title('Total Profit Percentage by Ticker', fontsize=16)
    plt.xlabel('Total Profit Percentage (%)', fontsize=12)
    plt.ylabel('Ticker', fontsize=12)
    plt.tight_layout()
    pdf.savefig()  # Save the figure
    plt.close()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Step 2: Supertrend Calculation Function
def calculate_supertrend(data, period=1400, multiplier=40):
    """
    Calculate the Supertrend indicator based on the LTP (Last Traded Price).
    
    Parameters:
        data (pd.DataFrame): DataFrame with a column 'LTP'.
        period (int): Lookback period for ATR calculation.
        multiplier (float): Multiplier for the ATR.
        
    Returns:
        pd.DataFrame: DataFrame with 'Supertrend' and 'Trend' columns.
    """
    data['high'] = data['LTP']  # Assuming LTP serves as both high and low
    data['low'] = data['LTP']
    data['close'] = data['LTP']
    
    # ATR calculation
    data['tr1'] = abs(data['high'] - data['low'])
    data['tr2'] = abs(data['high'] - data['close'].shift(1))
    data['tr3'] = abs(data['low'] - data['close'].shift(1))
    data['TR'] = data[['tr1', 'tr2', 'tr3']].max(axis=1)
    data['ATR'] = data['TR'].rolling(window=period).mean()
    
    # Calculate basic upper and lower bands
    data['UpperBand'] = ((data['high'] + data['low']) / 2) + (multiplier * data['ATR'])
    data['LowerBand'] = ((data['high'] + data['low']) / 2) - (multiplier * data['ATR'])
    
    # Supertrend calculation
    data['Supertrend'] = np.nan
    data['Trend'] = np.nan
    for i in range(1, len(data)):
        if data['close'][i] > data['UpperBand'][i - 1]:
            data.loc[i, 'Supertrend'] = data['LowerBand'][i]
            data.loc[i, 'Trend'] = 'uptrend'
        elif data['close'][i] < data['LowerBand'][i - 1]:
            data.loc[i, 'Supertrend'] = data['UpperBand'][i]
            data.loc[i, 'Trend'] = 'downtrend'
        else:
            data.loc[i, 'Supertrend'] = data['Supertrend'][i - 1]
            data.loc[i, 'Trend'] = data['Trend'][i - 1]

    return data

# Step 3: Apply Supertrend Calculation
data = calculate_supertrend(data)

# Step 4: Plot the Supertrend and Price
plt.figure(figsize=(12, 6))
plt.plot(data['Date_Time'], data['LTP'], label="LTP (Price)", alpha=0.7)
plt.plot(data['Date_Time'], data['Supertrend'], label="Supertrend", linewidth=2, color='orange')
plt.fill_between(data['Date_Time'], data['LTP'], data['Supertrend'], where=data['Trend'] == 'uptrend',
                 color='green', alpha=0.1, label='Uptrend')
plt.fill_between(data['Date_Time'], data['LTP'], data['Supertrend'], where=data['Trend'] == 'downtrend',
                 color='red', alpha=0.1, label='Downtrend')
plt.title("LTP with Supertrend Indicator")
plt.xlabel("Date_Time")
plt.ylabel("Price")
plt.legend()
plt.grid()
plt.show()


In [ ]:
import pandas as pd

# Step 1: Define Strategy 4A Function
def strategy_4A(data, threshold, aggressiveness, trailing_stop_n, trailing_stop_m):
    """
    Implements Strategy 4A for the provided dataset.
    """
    mode = 'none'
    trend = 'unknown'
    status = 'no_position'
    trailing_stop = None
    position = None
    trades = []

    for i in range(1, len(data)):
        current_price = data.loc[i, 'LTP']
        previous_price = data.loc[i - 1, 'LTP']

        # Determine trend
        if trend == 'unknown':
            if current_price >= (1 + 2 * threshold) * previous_price:
                trend = 'uptrend'
            elif current_price <= (1 - 2 * threshold) * previous_price:
                trend = 'downtrend'

        # Update trailing stop
        if status == 'holding' and trailing_stop is not None:
            if mode == 'up':
                trailing_stop = max(trailing_stop, current_price * (1 - threshold / (trailing_stop_n - trailing_stop_m)))
            elif mode == 'down':
                trailing_stop = min(trailing_stop, current_price * (1 + threshold / (trailing_stop_n - trailing_stop_m)))

        # Enter position
        if status == 'no_position':
            if current_price >= (1 + threshold) * previous_price:
                mode = 'up'
                position = current_price
                trailing_stop = current_price * (1 - threshold / trailing_stop_n)
                status = 'holding'
                trades.append({'entry_time': data.loc[i, 'Date_Time'], 'entry_price': current_price, 'type': 'long', 'PnL': None})
            elif current_price <= (1 - threshold) * previous_price:
                mode = 'down'
                position = current_price
                trailing_stop = current_price * (1 + threshold / trailing_stop_n)
                status = 'holding'
                trades.append({'entry_time': data.loc[i, 'Date_Time'], 'entry_price': current_price, 'type': 'short', 'PnL': None})

        # Exit position
        elif status == 'holding':
            if mode == 'up':
                if trend == 'uptrend' and current_price >= position * (1 + 2 * threshold + aggressiveness):
                    trades[-1].update({'exit_time': data.loc[i, 'Date_Time'], 'exit_price': current_price, 'PnL': current_price - position})
                    status = 'no_position'
                elif current_price <= trailing_stop:
                    trades[-1].update({'exit_time': data.loc[i, 'Date_Time'], 'exit_price': current_price, 'PnL': current_price - position})
                    status = 'no_position'

            elif mode == 'down':
                if trend == 'downtrend' and current_price <= position * (1 - 2 * threshold - aggressiveness):
                    trades[-1].update({'exit_time': data.loc[i, 'Date_Time'], 'exit_price': current_price, 'PnL': position - current_price})
                    status = 'no_position'
                elif current_price >= trailing_stop:
                    trades[-1].update({'exit_time': data.loc[i, 'Date_Time'], 'exit_price': current_price, 'PnL': position - current_price})
                    status = 'no_position'

    # Check if trades were executed
    if not trades:
        print("No trades executed for this parameter combination.")
    else:
        print(f"{len(trades)} trades executed.")

    return pd.DataFrame(trades)

# Step 2: Define the Test Combination Function
def test_combination(data, ticker, threshold, aggressiveness, trailing_stop_n, trailing_stop_m):
    """
    Test a single parameter combination on the specified ticker's data.
    """
    ticker_data = data[data['Ticker'] == ticker].reset_index(drop=True)
    trades = strategy_4A(ticker_data, threshold, aggressiveness, trailing_stop_n, trailing_stop_m)
    if trades.empty:
        print(f"No trades executed for Ticker: {ticker}, Threshold: {threshold}")
    total_pnl = trades['PnL'].sum() if 'PnL' in trades.columns else 0
    return {
        'Ticker': ticker,
        'Threshold': threshold,
        'Aggressiveness': aggressiveness,
        'Trailing Stop N': trailing_stop_n,
        'Trailing Stop M': trailing_stop_m,
        'Total PnL': total_pnl
    }

# Step 3: Robustness Testing Function (Sequential)
def robustness_testing(data, tickers, thresholds, aggressiveness_list, trailing_stop_n_values, trailing_stop_m_values):
    """
    Runs robustness testing sequentially across various parameters and tickers.
    """
    results = []
    for ticker in tickers:
        for threshold in thresholds:
            for aggressiveness in aggressiveness_list:
                for trailing_stop_n in trailing_stop_n_values:
                    for trailing_stop_m in trailing_stop_m_values:
                        result = test_combination(data, ticker, threshold, aggressiveness, trailing_stop_n, trailing_stop_m)
                        results.append(result)
    return pd.DataFrame(results)

# Main Execution Block
if __name__ == "__main__":
    # Load data into a DataFrame
    try:
        data['Date_Time'] = pd.to_datetime(data['Date_Time'])  # Convert Date_Time to datetime if not already

        # Parameters
        tickers = data['Ticker'].unique()
        thresholds = [0.008, 0.009, 0.01]
        aggressiveness_list = [0.0005]
        trailing_stop_n_values = [1]
        trailing_stop_m_values = [0.25]

        # Run robustness testing
        robustness_results = robustness_testing(data, tickers, thresholds, aggressiveness_list, trailing_stop_n_values, trailing_stop_m_values)
        print(robustness_results)

        # Export results to CSV
        robustness_results.to_csv('robustness_results.csv', index=False)
        print("Results successfully saved to 'robustness_results.csv'.")

    except Exception as e:
        print(f"An error occurred: {e}")
